In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path + "/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df.drop(columns=['Order_ID']).copy()
df_clean.head()

In [ ]:
# Task 2: Write your code here:
# we have missing values in: weather (categorical), traffic level (categorical), time of day (cat), courier experience years (num), delivery time (num)
# since delivery time is our target we need to drop the rows with missing delivery time
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())
df_clean = df_clean.dropna(subset=['Delivery_Time'])

df_clean.info()
df_clean.describe()

In [ ]:
# Task 3: Write your code here:
df_clean = df_clean.drop_duplicates()
df_clean.info()

In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

#i thought i saved the file that had the one hot label encoder :(

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()

df_clean[features] = scaler.fit_transform(df_clean[features])

df_clean.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
X = df_clean[features]
y = df_clean["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from tqdm import tqdm
def gradient_descent(X, y, learning_rate, n_iters=1000):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
maes = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  maes.append(mae)

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(maes, label='Train Loss')
plt.title('Loss over folds')
plt.xlabel('Epoch')
plt.ylabel('Loss (MAE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: